# Banking Support AI — RAG

## Architecture

Customer Query
→ BERT Intent Classifier
→ Document Retrieval (FAISS)
→ Relevant Banking Context
→ LLM
→ Grounded Support Response

This project reuses the previously trained Banking77 BERT classifier.
No retraining is performed.

In [1]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 23.1 MB/s eta 0:00:00


In [2]:
import os
import torch
import numpy as np
import pandas as pd
import faiss

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM
)

from sentence_transformers import SentenceTransformer

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
BERT_PATH = "/content/drive/MyDrive/banking77-bert-final"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_PATH)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_PATH)

bert_model.to("cuda")
bert_model.eval()

print("BERT loaded successfully.")
print("Number of intents:", bert_model.config.num_labels)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BERT loaded successfully.
Number of intents: 77


In [6]:
def classify_intent(query):
    inputs = bert_tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)[0]

    predicted_id = torch.argmax(probabilities).item()

    intent = bert_model.config.id2label[predicted_id]
    confidence = probabilities[predicted_id].item()

    return intent, confidence

In [7]:
classify_intent("My card payment was declined")

('declined_card_payment', 0.8030624389648438)

In [8]:
knowledge_base = [
    {
        "title": "Card arrival",
        "intent": "card_arrival",
        "content": """
        If a customer is waiting for a physical bank card, they can check
        the card delivery status through the bank's official app or support
        channel. If the expected delivery period has passed, the customer
        should contact the bank through an official support channel.
        Do not assume a specific delivery deadline unless the bank has
        provided one.
        """
    },

    {
        "title": "Card not working",
        "intent": "card_not_working",
        "content": """
        If a bank card is not working, the customer can check whether the
        card is active, whether it has expired, and whether the issue occurs
        at multiple merchants or terminals. If the card continues to fail,
        the customer should contact the bank through its official support
        channel.
        """
    },

    {
        "title": "Declined card payment",
        "intent": "declined_card_payment",
        "content": """
        A card payment can be declined for different reasons, including
        insufficient available funds, security checks, merchant issues,
        card restrictions, or technical problems. The customer should check
        the transaction details in the banking app and contact the bank if
        the reason is unclear.
        """
    },

    {
        "title": "Lost or stolen card",
        "intent": "lost_or_stolen_card",
        "content": """
        If a customer has lost a bank card or believes it was stolen, they
        should immediately use the bank's official app or contact the bank
        through an official support channel to freeze or block the card.
        Customers should not share their PIN, password, or OTP with anyone.
        """
    },

    {
        "title": "Change PIN",
        "intent": "change_pin",
        "content": """
        Customers should use the bank's official app, ATM, or other
        officially supported method to change their card PIN. The exact
        process depends on the bank. Customers should never share their PIN
        with support staff or other people.
        """
    },

    {
        "title": "Activate card",
        "intent": "activate_my_card",
        "content": """
        A new bank card may need to be activated before it can be used.
        Customers should follow the activation instructions provided by
        their bank, such as using the official banking app or another
        officially supported method.
        """
    },

    {
        "title": "Pending transfer",
        "intent": "pending_transfer",
        "content": """
        A transfer marked as pending has not yet reached a completed state.
        Customers should check the transaction status in their banking app.
        If the transfer remains pending longer than the bank's stated
        processing period, they should contact official bank support.
        """
    },

    {
        "title": "Failed transfer",
        "intent": "failed_transfer",
        "content": """
        A failed transfer means the transaction was not completed.
        Customers should review the transaction details and any error
        message shown by the bank. If the reason is unclear, they should
        contact the bank through an official support channel.
        """
    },

    {
        "title": "Declined transfer",
        "intent": "declined_transfer",
        "content": """
        A transfer may be declined because of account restrictions,
        security checks, insufficient available funds, incorrect details,
        or technical issues. Customers should review the transaction
        information and contact official bank support if they need further
        clarification.
        """
    },

    {
        "title": "Refund not showing",
        "intent": "Refund_not_showing_up",
        "content": """
        If a customer is expecting a refund that has not appeared, they
        should first check the original transaction and contact the merchant
        if necessary. The customer can also check the bank transaction
        history. Exact refund processing times depend on the merchant,
        payment method, and bank, so a specific deadline should not be
        assumed without verified information.
        """
    },

    {
        "title": "Unrecognized cash withdrawal",
        "intent": "cash_withdrawal_not_recognised",
        "content": """
        If a customer does not recognize a cash withdrawal, they should
        review the transaction details and immediately contact their bank
        through an official support channel. If the card may be compromised,
        they should follow the bank's official procedure for securing or
        blocking the card.
        """
    },

    {
        "title": "Exchange rate",
        "intent": "exchange_rate",
        "content": """
        Exchange rates can change over time and may differ between providers
        and transaction types. Customers should check the current exchange
        rate shown by their bank or official banking service before making
        a transaction.
        """
    },

    {
        "title": "Top up by card charge",
        "intent": "top_up_by_card_charge",
        "content": """
        A card top-up may involve charges depending on the bank, card,
        payment method, or account type. Customers should check the fee
        information provided by their bank before assuming that a particular
        fee applies.
        """
    },

    {
        "title": "Wrong cash amount",
        "intent": "wrong_amount_of_cash_received",
        "content": """
        If an ATM dispenses an incorrect amount of cash, the customer should
        keep the transaction details and contact the bank through its official
        support channel as soon as possible. They should not assume a
        particular resolution time without information from the bank.
        """
    },

    {
        "title": "Identity verification",
        "intent": "verify_my_identity",
        "content": """
        Identity verification may be required for certain banking services.
        Customers should complete verification only through the bank's
        official app, website, or another officially provided channel.
        Sensitive documents and personal information should not be shared
        through unofficial channels.
        """
    }
]

print("Knowledge documents:", len(knowledge_base))

Knowledge documents: 15


In [9]:
knowledge_df = pd.DataFrame(knowledge_base)

knowledge_df

,title,intent,content
0,Card arrival,card_arrival,\n If a customer is waiting for a physi...
1,Card not working,card_not_working,"\n If a bank card is not working, the c..."
2,Declined card payment,declined_card_payment,\n A card payment can be declined for d...
3,Lost or stolen card,lost_or_stolen_card,\n If a customer has lost a bank card o...
4,Change PIN,change_pin,\n Customers should use the bank's offi...
5,Activate card,activate_my_card,\n A new bank card may need to be activ...
6,Pending transfer,pending_transfer,\n A transfer marked as pending has not...
7,Failed transfer,failed_transfer,\n A failed transfer means the transact...
8,Declined transfer,declined_transfer,\n A transfer may be declined because o...
9,Refund not showing,Refund_not_showing_up,\n If a customer is expecting a refund ...


In [10]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

documents = knowledge_df["content"].tolist()

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", document_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (15, 384)


In [11]:
dimension = document_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(document_embeddings)

print("Documents stored in FAISS:", index.ntotal)

Documents stored in FAISS: 15


In [12]:
def retrieve_documents(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "title": knowledge_df.iloc[idx]["title"],
            "intent": knowledge_df.iloc[idx]["intent"],
            "content": knowledge_df.iloc[idx]["content"].strip(),
            "score": float(score)
        })

    return results

In [13]:
query = "My card payment keeps getting rejected"

results = retrieve_documents(query, top_k=3)

for result in results:
    print("=" * 60)
    print("Title:", result["title"])
    print("Intent:", result["intent"])
    print("Similarity:", round(result["score"], 3))
    print(result["content"])

Title: Declined card payment
Intent: declined_card_payment
Similarity: 0.616
A card payment can be declined for different reasons, including
        insufficient available funds, security checks, merchant issues,
        card restrictions, or technical problems. The customer should check
        the transaction details in the banking app and contact the bank if
        the reason is unclear.
Title: Unrecognized cash withdrawal
Intent: cash_withdrawal_not_recognised
Similarity: 0.518
If a customer does not recognize a cash withdrawal, they should
        review the transaction details and immediately contact their bank
        through an official support channel. If the card may be compromised,
        they should follow the bank's official procedure for securing or
        blocking the card.
Title: Card not working
Intent: card_not_working
Similarity: 0.437
If a bank card is not working, the customer can check whether the
        card is active, whether it has expired, and whether the 

In [14]:
LLM_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16
)

llm_model.to("cuda")
llm_model.eval()

print("TinyLlama loaded.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama loaded.


In [30]:
def generate_rag_response(query, top_k=3):

    # 1. BERT classification
    intent, confidence = classify_intent(query)

    # 2. Retrieve relevant knowledge
    retrieved = retrieve_documents(query, top_k=top_k)

    context = "\n\n".join(
        [
            f"Document: {item['title']}\n{item['content']}"
            for item in retrieved
        ]
    )

    # 3. Build the chat messages
    messages = [
        {
            "role": "system",
            "content": (
                "You are a banking support assistant. "
                "Answer the customer's question using the provided knowledge context. "
                "Do not invent bank policies, fees, deadlines, balances, or requirements. "
                "Give only the final answer to the customer."
            )
        },
        {
            "role": "user",
            "content": (
                f"Knowledge context:\n{context}\n\n"
                f"Customer question: {query}"
            )
        }
    ]

    # 4. Tokenize using TinyLlama's official chat-template method
    inputs = llm_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")

    # Remove the model's default max_length so only max_new_tokens is used
    llm_model.generation_config.max_length = None

    # 5. Generate response
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.3,
            top_k=40,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    # 6. Decode only newly generated tokens
    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]

    response = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "intent": intent,
        "confidence": confidence,
        "retrieved_documents": retrieved,
        "response": response
    }

In [31]:
test_queries = [
    "My card payment was declined",
    "I lost my card",
    "My transfer is still pending",
    "The ATM gave me the wrong amount of cash",
    "I don't recognize this cash withdrawal"
]

for query in test_queries:

    result = generate_rag_response(query)

    print("=" * 70)
    print("QUERY:", query)
    print("INTENT:", result["intent"])
    print("CONFIDENCE:", round(result["confidence"], 3))
    print("\nRETRIEVED:")

    for doc in result["retrieved_documents"][:2]:
        print("-", doc["title"], "| score:", round(doc["score"], 3))

    print("\nRESPONSE:")
    print(result["response"])

QUERY: My card payment was declined
INTENT: declined_card_payment
CONFIDENCE: 0.803

RETRIEVED:
- Declined card payment | score: 0.78
- Declined transfer | score: 0.531

RESPONSE:
Based on the provided knowledge context, the customer's question is:

"My card payment was declined. Can you please provide me with the final answer to this question?"

The final answer to this question is:

If the reason for the decline is insufficient available funds, the customer should check their transaction details in the banking app and contact the bank if the reason is unclear. If the transaction is for an account restriction, the customer should review the
QUERY: I lost my card
INTENT: lost_or_stolen_card
CONFIDENCE: 0.524

RETRIEVED:
- Lost or stolen card | score: 0.533
- Unrecognized cash withdrawal | score: 0.444

RESPONSE:
Based on the provided knowledge context, the customer's question is: "If I lost my bank card, what should I do?"

The final answer is:

1. If a customer has lost a bank card, t

In [32]:
query = "My refund hasn't appeared yet"

result = generate_rag_response(query)

print("CUSTOMER QUERY")
print(query)

print("\nBERT INTENT")
print(result["intent"])

print("\nBERT CONFIDENCE")
print(round(result["confidence"], 3))

print("\nRETRIEVED KNOWLEDGE")
for doc in result["retrieved_documents"]:
    print("\n---", doc["title"], "---")
    print(doc["content"])

print("\nGENERATED RESPONSE")
print(result["response"])

CUSTOMER QUERY
My refund hasn't appeared yet

BERT INTENT
Refund_not_showing_up

BERT CONFIDENCE
0.847

RETRIEVED KNOWLEDGE

--- Refund not showing ---
If a customer is expecting a refund that has not appeared, they
        should first check the original transaction and contact the merchant
        if necessary. The customer can also check the bank transaction
        history. Exact refund processing times depend on the merchant,
        payment method, and bank, so a specific deadline should not be
        assumed without verified information.

--- Card arrival ---
If a customer is waiting for a physical bank card, they can check
        the card delivery status through the bank's official app or support
        channel. If the expected delivery period has passed, the customer
        should contact the bank through an official support channel.
        Do not assume a specific delivery deadline unless the bank has
        provided one.

--- Pending transfer ---
A transfer marked as p

In [33]:
evaluation_queries = [
    ("My card payment was rejected", "declined_card_payment"),
    ("I lost my card", "lost_or_stolen_card"),
    ("My bank transfer is pending", "pending_transfer"),
    ("ATM gave me less cash than requested", "wrong_amount_of_cash_received"),
    ("I don't recognize a cash withdrawal", "cash_withdrawal_not_recognised"),
    ("I need to change my PIN", "change_pin"),
    ("My card hasn't arrived", "card_arrival"),
]

correct = 0

for query, expected_intent in evaluation_queries:

    results = retrieve_documents(query, top_k=3)

    retrieved_intents = [
        result["intent"]
        for result in results
    ]

    hit = expected_intent in retrieved_intents

    if hit:
        correct += 1

    print(
        f"{'✓' if hit else '✗'} "
        f"{query} → {retrieved_intents}"
    )

print("\nTop-3 retrieval accuracy:",
      f"{correct / len(evaluation_queries):.1%}")

✓ My card payment was rejected → ['declined_card_payment', 'cash_withdrawal_not_recognised', 'lost_or_stolen_card']
✓ I lost my card → ['lost_or_stolen_card', 'cash_withdrawal_not_recognised', 'card_not_working']
✓ My bank transfer is pending → ['pending_transfer', 'declined_transfer', 'failed_transfer']
✓ ATM gave me less cash than requested → ['wrong_amount_of_cash_received', 'cash_withdrawal_not_recognised', 'declined_card_payment']
✓ I don't recognize a cash withdrawal → ['cash_withdrawal_not_recognised', 'wrong_amount_of_cash_received', 'declined_transfer']
✓ I need to change my PIN → ['change_pin', 'lost_or_stolen_card', 'activate_my_card']
✓ My card hasn't arrived → ['card_arrival', 'activate_my_card', 'card_not_working']

Top-3 retrieval accuracy: 100.0%


In [34]:
query = "Someone took cash from my account and I don't recognize it"

result = generate_rag_response(query)

print("CUSTOMER")
print(query)

print("\nBERT")
print(f"Intent: {result['intent']}")
print(f"Confidence: {result['confidence']:.2%}")

print("\nRAG")
for doc in result["retrieved_documents"]:
    print(
        f"- {doc['title']} "
        f"(similarity={doc['score']:.3f})"
    )

print("\nLLM RESPONSE")
print(result["response"])

CUSTOMER
Someone took cash from my account and I don't recognize it

BERT
Intent: cash_withdrawal_not_recognised
Confidence: 85.62%

RAG
- Unrecognized cash withdrawal (similarity=0.638)
- Wrong cash amount (similarity=0.496)
- Lost or stolen card (similarity=0.424)

LLM RESPONSE
Knowledge context:
Document: Unrecognized cash withdrawal

Customer question: Someone took cash from my account and I don't recognize it.

Answer:
1. Review the transaction details and immediately contact your bank through an official support channel.
2. If the card may be compromised, follow the bank's official procedure for securing or blocking the card.
3. If an ATM dispenses an incorrect amount of cash
